# PneumoAI — chạy toàn bộ thí nghiệm cuối kỳ

**Trước khi bấm Run All** (cột bên phải của Kaggle):
1. **Accelerator: GPU** (P100 hoặc T4) và **Internet: On** (để clone repo + tải trọng số ResNet18).
2. **Add Input** → dataset `vaillant/siim-ptx-png`.
3. *(Nên có)* Add Input → output của notebook cũ có các file `*_best.pt` (E1–E4). Không có thì notebook tự train lại E1–E4 (thêm ~2–3 giờ).

Notebook tự áp code nửa sau (kể cả khi chưa push lên GitHub), chạy test, kiểm tra rò rỉ, dò ngưỡng trên **Val**, chấm **Test 1 lần**,
rồi gom tất cả vào **`/kaggle/working/final_results.zip`** → tải file đó về gửi lại.
Nếu bị ngắt giữa chừng: bấm Run All lại, bước nào xong rồi sẽ được bỏ qua (khi giữ nguyên `/kaggle/working`).

In [ ]:
import os, subprocess, base64, pathlib
os.chdir('/kaggle/working')
if not pathlib.Path('pneumothorax-detection').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/hungt58/pneumothorax-detection'], check=True)
os.chdir('/kaggle/working/pneumothorax-detection')
subprocess.run(['git', 'checkout', '-q', 'ChuongDinh'], check=True)
subprocess.run(['git', 'pull', '-q', 'origin', 'ChuongDinh'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
# Code nửa sau (patch nhúng sẵn) - chỉ áp nếu repo chưa có
PATCH_B64 = (
    "..."
    "X18iOgorICAgIG1haW4oKQotLSAKMi40My4wCgo="
)
if not pathlib.Path('run_final.py').exists():
    pathlib.Path('/kaggle/working/nua_sau.patch').write_bytes(base64.b64decode(PATCH_B64))
    git = ['git', '-c', 'user.name=Chuongdinh', '-c', 'user.email=quocchuong277@gmail.com']
    r = subprocess.run(git + ['am', '/kaggle/working/nua_sau.patch'])
    for _ in range(3):  # commit nào đã có trên GitHub thì bỏ qua
        if r.returncode == 0:
            break
        r = subprocess.run(git + ['am', '--skip'])
    assert pathlib.Path('run_final.py').exists(), 'Áp patch lỗi - gửi log cho t'
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
print('run_final.py:', pathlib.Path('run_final.py').exists())

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
# Tự tìm thư mục ảnh/mask của SIIM
import glob
IMAGES = next(p for p in glob.glob('/kaggle/input/**/images', recursive=True)
              if os.path.isdir(p.replace('images', 'segmentations')))
MASKS = IMAGES.replace('images', 'segmentations')
print(IMAGES, len(os.listdir(IMAGES)), 'ảnh')
print(MASKS, len(os.listdir(MASKS)), 'mask')
print('Checkpoint tìm thấy:', glob.glob('/kaggle/input/**/*_best.pt', recursive=True))

In [ ]:
# Chạy hết: rò rỉ -> E1-E4 -> E2+E4 -> E4b -> E2@512 -> ảnh minh họa (vài giờ)
!python run_final.py --images "{IMAGES}" --masks "{MASKS}" --ckpt-search /kaggle/input

In [ ]:
from IPython.display import Markdown, FileLink
display(Markdown(open('results/final_summary.md', encoding='utf-8').read()))
FileLink('/kaggle/working/final_results.zip')